In [207]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
import calendar
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [208]:
# read excel sheet with a password  
def read_protected_excel(file_path, password, sheet_names, skiprows_list):  
    """Read multiple sheets from a password-protected Excel file."""  
    with open(file_path, 'rb') as f:  
        office_file = msoffcrypto.OfficeFile(f)  
        office_file.load_key(password=password)  
        decrypted = io.BytesIO()  
        office_file.decrypt(decrypted)

    dfs = []  
    for sheet_name, skiprows in zip(sheet_names, skiprows_list):  
        decrypted.seek(0)  
        df = pd.read_excel(decrypted, sheet_name=sheet_name, skiprows=skiprows)  
        dfs.append(df)

    return dfs  

# set the file path for manpower files
base_path = 'L:/2026 Pilar Plant Files/2026 Manpower Detail/'  
password = 'bamhorse2'  
# set the tabs that are needed
sheet_names = ['PPE LDR', 'PPE FTEs']  
# add the number of rows to skip at the top of each sheet (to ignore headers)
skiprows_list = [6, 8]
# specify required filename keyword  
required_name = 'LENOX SAL'


def get_biweekly_dates(start_date, end_date):  
    biweekly_dates = []  
    current = start_date  
    while current <= end_date:  
        last_day = calendar.monthrange(current.year, current.month)[1]  
        if current.day != 15 and current.day != last_day:  
            biweekly_dates.append(current)  
        current += pd.Timedelta(days=1)  
    return biweekly_dates


# get all matching files in the directory  
all_files = glob.glob(os.path.join(base_path, '*.xlsm'))

# filter for required name and extract date from filename  
all_biweekly_files = []  
for f in all_files:  
    filename = os.path.basename(f)  
    # only include files that have the required name  
    if required_name.upper() not in filename.upper():  
        continue

    # extract date in YYYY-MM-DD format from filename  
    date_match = re.search(r'(\d{4}-\d{2}-\d{2})', filename)  
    if date_match:  
        date_str = date_match.group(1)  
        file_date = pd.to_datetime(date_str)  
        # check if it's NOT the 15th or last day of the month  
        last_day = calendar.monthrange(file_date.year, file_date.month)[1]  
        # exception: always include May 31  
        is_may_31 = (file_date.month == 5 and file_date.day == 31)  
        if is_may_31 or (file_date.day != 15 and file_date.day != last_day):  
            all_biweekly_files.append((f, date_str))  
    else:  
        print(f"  WARNING: Could not parse date from {filename}")

print(f"\nBiweekly files found: {len(all_biweekly_files)}")  
for f, d in all_biweekly_files:  
    print(f"  {os.path.basename(f)} (date: {d})")


# combine into one list with pay type labels  
files = [(f, 'Biweekly', d) for f, d in all_biweekly_files]

all_ppe_ldr = []
all_ppe_ftes = []

# for loop!
for file_path, pay_type, date in files:  
    # extract site name from filename (everything before the date)  
    filename = os.path.basename(file_path)  
    site = filename.split(' ')[0].strip()  
    
    print(f"\nReading {pay_type} | Site: {site} | Date: {date}")

    try:  
        dfs = read_protected_excel(file_path, password, sheet_names, skiprows_list)
        for df in dfs:
            # adding extra columns  
            df['Pay_Type'] = pay_type  
            df['Report_Date'] = pd.to_datetime(date)  
            df['BU'] = site

        # appending data to dataframe
        all_ppe_ldr.append(dfs[0])  
        all_ppe_ftes.append(dfs[1])  

    except Exception as e:  
        print(f"  ERROR: {e}")


# combines lists of dataframes into one dataframe  
ppe_ldr_combined = pd.concat(all_ppe_ldr, ignore_index=True)
ppe_ftes_combined = pd.concat(all_ppe_ftes, ignore_index=True)

# renames columns to preferred format 
ppe_ftes_combined.columns = ppe_ftes_combined.columns.str.strip().str.replace(r'[^a-zA-Z0-9]', '_', regex=True).str.replace(r'_+', '_', regex=True).str.strip('_').str.lower()  
ppe_ldr_combined.columns = (  
    ppe_ldr_combined.columns.str.strip()  
    .str.replace(r'[^a-zA-Z0-9]', '_', regex=True)  
    .str.replace(r'_+', '_', regex=True)  
    .str.strip('_')  
    .str.lower()  
)

print(f"\n--- Summary ---")
print(f"PPE FTEs: {ppe_ftes_combined.shape}")  
print(f"PPE LDR:  {ppe_ldr_combined.shape}")  
print(f"Sites found: {ppe_ldr_combined['bu'].unique()}")  


Biweekly files found: 19
  LENOX SALAERN LENOX_EXEC 2026-01-10.xlsm (date: 2026-01-10)
  LENOX SALAERN LENOX_EXEC 2026-01-24.xlsm (date: 2026-01-24)
  LENOX SALAERN LENOX_EXEC 2026-02-07.xlsm (date: 2026-02-07)
  LENOX SALAERN LENOX_EXEC 2026-02-21.xlsm (date: 2026-02-21)
  LENOX SALAERN LENOX_EXEC 2026-03-07.xlsm (date: 2026-03-07)
  LENOX SALAERN LENOX_EXEC 2026-04-04.xlsm (date: 2026-04-04)
  LENOX SALAERN LENOX_EXEC 2026-04-18.xlsm (date: 2026-04-18)
  LENOX SALAERN LENOX_EXEC 2026-05-02.xlsm (date: 2026-05-02)
  LENOX SALAERN LENOX_EXEC 2026-05-16.xlsm (date: 2026-05-16)
  LENOX SALAERN LENOX_EXEC 2026-05-31.xlsm (date: 2026-05-31)
  LENOX SALAERN LENOX_EXEC 2026-06-13.xlsm (date: 2026-06-13)
  LENOX SALAERN LENOX_EXEC 2026-06-27.xlsm (date: 2026-06-27)
  LENOX SALAERN LENOX_EXEC 2026-07-11.xlsm (date: 2026-07-11)
  LENOX SALAERN LENOX_EXEC 2026-07-25.xlsm (date: 2026-07-25)
  LENOX SALAERN LENOX_EXEC 2026-08-08.xlsm (date: 2026-08-08)
  LENOX SALAERN LENOX_EXEC 2026-08-22.xlsm (

In [209]:
ppe_ldrs = ppe_ldr_combined[ppe_ldr_combined['gl_deptid'].notna()]

ppe_ldr_grouped = ppe_ldrs.groupby(['gl_deptid', 'description', 'jobcode', 'description_1', 'name', 'emplid', 'hr_deptid', 'description_2', 'position',
                                    'full_part', 'reg_temp', 'pay_end_dt', 'pay_type', 'report_date', 'bu'])[['reg_dollars',
                                    'ot_dollars', 'hol_wkd_dollars', 'payout_dollars', 'other_dollars', 'non_prod_dollars', 'total_dollars', 'regular_hours',
                                    'ot_hours', 'hol_wkd_hours', 'non_prod_hours', 'other_hours', 'payout_hours', 'total_hours']].sum().reset_index()

In [210]:
ppe_ldr_grouped['true_reg_dollars'] = ppe_ldr_grouped['total_dollars'] - ppe_ldr_grouped['ot_dollars']
ppe_ldr_grouped['true_reg_hours'] = ppe_ldr_grouped['total_hours'] - ppe_ldr_grouped['ot_hours']

ppe_ldr = ppe_ldr_grouped[['bu', 'report_date', 'gl_deptid', 'description', 'jobcode', 'description_1', 'name', 'emplid', 'hr_deptid', 'description_2', 'position',
                            'full_part', 'reg_temp', 'pay_end_dt', 'pay_type','true_reg_dollars', 'ot_dollars', 'total_dollars', 'true_reg_hours', 'ot_hours', 'total_hours']]

pc_lookup = pd.read_excel('C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/pc_fte_lookup.xlsx')

In [211]:
ppe_hours = pd.merge(ppe_ldr, pc_lookup[['Emplid', 'STD Hrs']], how='left', left_on='emplid', right_on='Emplid')

ppe_empl_active = ppe_hours[ppe_hours['Emplid'].notna()]

In [212]:
year_grouped = ppe_empl_active.groupby(['name', 'emplid'])[['true_reg_dollars', 'ot_dollars', 'total_dollars', 'true_reg_hours', 'ot_hours', 
                                                            'total_hours', 'STD Hrs']].sum().reset_index()

year_grouped['reg_ftes'] = year_grouped['true_reg_hours'] / (year_grouped['STD Hrs']*2)
year_grouped['ot_ftes'] = year_grouped['ot_hours'] / (year_grouped['STD Hrs']*2)
year_grouped['total_ftes'] = year_grouped['total_hours'] / (year_grouped['STD Hrs']*2)

year_grouped['ot_pct'] = year_grouped['ot_ftes'] / year_grouped['total_ftes']
year_grouped['ot_flag'] = year_grouped['ot_pct'] >= 0.25

ot_hounds = year_grouped[year_grouped['ot_flag'] == True].sort_values(by='ot_pct', ascending=False)

display(ot_hounds)

,name,emplid,true_reg_dollars,ot_dollars,total_dollars,true_reg_hours,ot_hours,total_hours,STD Hrs,reg_ftes,ot_ftes,total_ftes,ot_pct,ot_flag
2079,"Phillander,Delon C.",157038.0,42479.36,61677.25,104156.61,1275.00,1231.50,2506.50,637.5,1.000000,0.965882,1.965882,0.491323,True
985,"Gonta,Viktor",158414.0,104998.62,118925.51,223924.13,1201.25,1053.25,2254.50,595.0,1.009454,0.885084,1.894538,0.467177,True
1009,"Grant,Andre A",206092.0,41227.89,53379.03,94606.92,1282.50,1119.80,2402.30,637.5,1.005882,0.878275,1.884157,0.466137,True
166,"Bah,Ibrahima S",232287.0,35339.57,48315.77,83655.34,1275.00,1104.50,2379.50,637.5,1.000000,0.866275,1.866275,0.464173,True
1057,"Gyamfi,Dorcas Z",214905.0,46765.45,58319.81,105085.26,1245.00,1049.50,2294.50,637.5,0.976471,0.823137,1.799608,0.457398,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2732,"Veliz,Omar A",146077.0,60027.58,30223.54,90251.12,1282.50,432.75,1715.25,937.5,0.684000,0.230800,0.914800,0.252296,True
2865,"Wright,Tamisha",199199.0,34899.28,17676.40,52575.68,1276.43,429.50,1705.93,637.5,1.001122,0.336863,1.337984,0.251769,True
2691,"Ukpong,Ndifreke",582362.0,41646.78,20782.43,62429.21,1266.50,423.75,1690.25,1162.5,0.544731,0.182258,0.726989,0.250703,True
532,"Commedo,Dennis L",165821.0,36278.31,19173.51,55451.82,1282.75,428.41,1711.16,637.5,1.006078,0.336008,1.342086,0.250362,True


In [213]:
empl_ot_raw = ppe_empl_active[ppe_empl_active['emplid'].isin(ot_hounds['emplid'])]

empl_by_pp = empl_ot_raw.groupby(  
    ['name', 'emplid', 'report_date', 'STD Hrs']  
)[['true_reg_dollars', 'ot_dollars', 'total_dollars',  
   'true_reg_hours', 'ot_hours', 'total_hours']].sum().reset_index()

empl_by_pp_copy = empl_by_pp.copy()

empl_by_pp['reg_ftes']   = empl_by_pp['true_reg_hours'] / (empl_by_pp['STD Hrs'] * 2)  
empl_by_pp['ot_ftes']    = empl_by_pp['ot_hours']       / (empl_by_pp['STD Hrs'] * 2)  
empl_by_pp['total_ftes'] = empl_by_pp['total_hours']    / (empl_by_pp['STD Hrs'] * 2)

empl_index_cols = ['name', 'emplid']  
duplicates = empl_by_pp[empl_by_pp.duplicated(subset=empl_index_cols + ['report_date'], keep=False)]  
print(f"Duplicate rows: {len(duplicates)}")  
if len(duplicates) > 0:  
    display(duplicates)  
    empl_by_pp = empl_by_pp.drop_duplicates(subset=empl_index_cols + ['report_date'], keep='first')

empl_pivot = pd.pivot_table(  
    empl_by_pp,  
    values='ot_ftes',  
    index=empl_index_cols,  
    columns=['report_date'],  
    aggfunc='sum'  
).reset_index().sort_values(by='name')

empl_pivot.columns = [  
    col.strftime('%Y-%m-%d') if isinstance(col, pd.Timestamp) else col  
    for col in empl_pivot.columns  
]  
empl_date_cols = [col for col in empl_pivot.columns if col not in empl_index_cols]  
empl_pivot[empl_date_cols] = empl_pivot[empl_date_cols].fillna(0)

empl_pivot['total'] = empl_pivot[empl_date_cols].sum(axis=1)  
empl_pivot_nonzero = empl_pivot[empl_pivot['total'] != 0].drop(columns='total').sort_values(by='name')

empl_pivot_detail = pd.merge(  
    pc_lookup, empl_pivot_nonzero,  
    how='right',  
    left_on='Emplid',  
    right_on='emplid'  
)

empl_pivot_ytd = pd.merge(  
    empl_pivot_detail, ot_hounds[['emplid', 'ot_ftes']],  
    how='left',  
    on='emplid'  
)

vp_xwalk = pd.read_excel(  
    'L:/2026 Pilar Plant Files/Crosswalks/Manhattan Region Crosswalk.xlsx',  
    sheet_name='vp_dept'  
)

empl_pivot_final = pd.merge(  
    vp_xwalk, empl_pivot_ytd,  
    how='right',  
    left_on='Dept',  
    right_on='Rpt Dept'  
).rename(columns={  
    'Rpt Dept': 'Dept ID',  
    'Rpt Dept Desc': 'Dept Name',  
    'Jobcode Title': 'Job Desc',  
    'name': 'Name',  
    'ot_ftes': 'YTD OT FTES'  
}).drop(columns=['Incumbent Name', 'STD Hrs', 'emplid', 'Dept'])

display(empl_pivot_final)  

Duplicate rows: 0


,VP,Dept ID,Dept Name,Job Code,Job Desc,Emplid,Name,2026-01-10,2026-01-24,2026-02-07,...,2026-05-16,2026-05-31,2026-06-13,2026-06-27,2026-07-11,2026-07-25,2026-08-08,2026-08-22,2026-09-05,YTD OT FTES
0,"Jurik, Christopher",15600360,O360 Managed Staff and OTS,116649,Access Service Representative,246502,"Abel,Natisha A",0.681600,0.640267,0.418400,...,0.640000,0.710400,0.636667,0.494933,0.100000,0.597067,0.197600,0.960267,1.238000,0.581341
1,"Cohen, Jill",15651050,POS - Central Services,108131,Sterile Processing Tech (LH),610465,"Abredu,Roland",0.000000,0.000000,0.076667,...,0.620000,0.450000,0.280000,0.776667,0.346667,0.606667,0.433333,0.520000,0.446667,0.366667
2,"Cole, Brandon L",15693010,Environmental Services,108143,Housekeeper,247901,"Abreu,Max",0.000000,0.790000,0.880000,...,0.676400,0.796667,0.799600,0.789600,0.792267,0.793333,0.800000,0.800000,0.850000,0.705652
3,"Browne, Kevin",15610030,NRSG - Float,115242,Patient Care Associate,237495,"Aheto,Vida",0.486667,0.293333,0.676667,...,0.000000,0.193333,0.386667,0.470000,0.300000,0.386667,0.003333,0.483333,0.386667,0.046434
4,"Cole, Brandon L",15693010,Environmental Services,108604,Waxer Stripper,159177,"Alston,Lynnasia",0.606667,0.276667,0.466667,...,0.200000,0.690000,0.308933,0.803333,0.690000,0.073333,0.200000,0.210000,0.840000,0.468215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,"Cole, Brandon L",15693010,Environmental Services,108143,Housekeeper,183124,"Williams,Tony",0.722933,0.734400,0.513333,...,0.706667,0.646667,0.620000,0.633333,0.646667,0.546667,0.546667,0.543333,0.553333,0.589200
158,"Cole, Brandon L",15650020,Lab-Chemistry,108410,Lead Path Technologist,159563,"Witkowski,Agnieszka",0.600000,0.998857,0.710714,...,0.750000,0.957143,0.950000,0.850000,0.471429,0.471429,0.557143,0.585714,0.578571,0.355744
159,"Cole, Brandon L",15691000,Sup Svc-Food and Nutrition,108142,Host/Hostess,199199,"Wright,Tamisha",0.396667,0.476667,0.330000,...,0.200000,0.396667,0.246667,0.276667,0.196667,0.300000,0.346667,0.293333,0.000000,0.336863
160,"Browne, Kevin",15613035,NRSG - 7 Uris,115242,Patient Care Associate,163817,"Wright-Smith,Muriel I",0.193333,0.383333,0.680000,...,0.386667,0.096667,0.293333,0.386667,0.200000,0.676667,0.193333,0.676667,0.490000,0.344737


In [214]:
# function to clean ftes tabs
def clean_ftes(sheet):
    # remove subtotal rowa
    sheet_c = sheet[~sheet['deptid'].str.contains('total', case=False, na=False)]

    # forward fill department ids and names
    sheet_c['deptid'] = sheet_c['deptid'].ffill()
    sheet_c['department_description'] = sheet_c['department_description'].ffill()

    # remove department ids that are blank
    sheet_n = sheet_c[sheet_c['deptid'] != '(blank)']
    # set type to integer
    sheet_n[['deptid', 'job_code']] = sheet_n[['deptid', 'job_code']].astype('Int64') 

    # remove corporate retained and employee health services
    sheet_clean = sheet_n[~sheet_n['department_description'].str.contains('Corporate Retained|Corp Retained|Employee Health Svcs', case=False, na=False)]  

    # fix duplicate column names  
    cols = pd.Series(sheet_clean.columns)  
    for dup in cols[cols.duplicated()].unique():  
        count = 0  
        for i in range(len(cols)):  
            if cols[i] == dup:  
                count += 1  
                if count > 1:  
                    cols[i] = f"{dup}_{count}"  
    sheet_clean.columns = cols
    return sheet_clean


ppe_ftes = clean_ftes(ppe_ftes_combined)
ot_ftes = ppe_ftes[['report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description', 'overtime']]


jc_index_cols = ['deptid', 'department_description', 'job_code', 'job_code_description']

jc_pivot = pd.pivot_table(  
    ot_ftes,  
    values='overtime',  
    index=jc_index_cols,  
    columns=['report_date'],  
    aggfunc='sum'  
).reset_index()


jc_pivot.columns = [  
    col.strftime('%Y-%m-%d') if isinstance(col, pd.Timestamp) else col  
    for col in jc_pivot.columns  
]  
jc_date_cols = [col for col in jc_pivot.columns if col not in jc_index_cols]  
jc_pivot[jc_date_cols] = jc_pivot[jc_date_cols].fillna(0)

jc_pivot['total'] = jc_pivot[jc_date_cols].sum(axis=1)  
jc_pivot_nonzero = jc_pivot[jc_pivot['total'] != 0].drop(columns='total').sort_values(by=['deptid', 'job_code_description'])

jc_pivot_vp = pd.merge(vp_xwalk, jc_pivot_nonzero, 
                       how='right', 
                       left_on='Dept', 
                       right_on='deptid'
                       ).rename(columns={
                           'vp': 'VP',
                           'deptid': 'Dept ID', 
                           'department_description': 'Dept Name',
                           'job_code': 'Job Code',
                           'job_code_description': 'Job Desc'
                           }).drop(columns='Dept')


vp_summary = jc_pivot_vp.groupby(['VP'])[jc_date_cols].sum().reset_index()

In [216]:
output_path = 'C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/OT_Analysis.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:  
    vp_summary.to_excel(writer, sheet_name='VP Summary', index=False)  
    jc_pivot_vp.to_excel(writer, sheet_name='Job Code Detail', index=False)  
    empl_pivot_final.to_excel(writer, sheet_name='Employee Detail', index=False)

print(f"Exported to {output_path}")  

Exported to C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/OT_Analysis.xlsx
